In [1]:
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient
from azure.storage.blob import BlobServiceClient
import io

In [2]:
%run ../Framework/AzureKeyVault.ipynb

In [3]:
class ADLSClient():
    def __init__(self):
        self.adls_container = KeyVault().get_secret("ADLSContainerName")
        self.service_client = self.create_service_client()
        self.blob_client = self.create_blob_client()
        
    def create_service_client(self):
        credential = DefaultAzureCredential()
        adls_url = KeyVault().get_secret("ADLSAccountURL")
        adls_account_key = KeyVault().get_secret("ADLSKey")
        return DataLakeServiceClient(account_url=adls_url, credential=credential)
        
    def create_blob_client(self):
        adls_connection_string = KeyVault().get_secret("ADLSConnectionString")
        return BlobServiceClient.from_connection_string(adls_connection_string)

    def create_container_client(self):
        return self.blob_client.get_container_client(self.adls_container)
    
    def create_file_system_client(self):
        return self.service_client.get_file_system_client(self.adls_container)

    def create_file_client(self, filepath):
        file_system_client = self.create_file_system_client()
        return file_system_client.get_file_client(filepath)
    
    def get_csv_file_data(self, filepath, sep):
        file_client = self.create_file_client(filepath)
        
        raw_data = file_client.download_file().readall()
        file_content_str = raw_data.decode("utf-8")
        
        file_like = io.StringIO(file_content_str)
        return pd.read_csv(file_like, sep=sep)